# 01 — Explore the Data

This notebook explores the BRONZE data layer:
- Salesforce CRM: Opportunities, Vehicles (Assets), Defects, Deliveries
- Odos Telemetry: Raw GPS/battery/speed events
- VIN overlap between the two systems


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA BRONZE;
USE WAREHOUSE HYDRAB_HOL_WH;

## Salesforce: Sales Pipeline

In [ ]:
-- Top opportunities by value
SELECT "Name", "StageName", "Amount", "CloseDate", "Region__c"
FROM OPPORTUNITY
WHERE "Amount" > 1000000
ORDER BY "Amount" DESC
LIMIT 20;

In [ ]:
-- Pipeline by stage
SELECT "StageName", 
       COUNT(*) as deals, 
       SUM("Amount") as total_value,
       ROUND(AVG("Amount"), 0) as avg_deal
FROM OPPORTUNITY
GROUP BY "StageName"
ORDER BY total_value DESC;

## Salesforce: Vehicle Fleet

In [ ]:
-- Fleet composition by model
SELECT "Name" as model,
       COUNT(*) as vehicles,
       COUNT(DISTINCT "AccountId") as customers
FROM ASSET
WHERE "Chassis_Number__c" IS NOT NULL
GROUP BY "Name"
ORDER BY vehicles DESC
LIMIT 15;

## Odos Telemetry: Raw Events

In [ ]:
-- Sample telemetry events (raw JSON)
SELECT *
FROM ODOS_EVENTS
LIMIT 5;

In [ ]:
-- Parse key signals from Odos JSON
-- The data is stored as VARCHAR; we parse it and flatten the signals array
SELECT 
  PARSE_JSON(RAW_JSON):vin::STRING as vin,
  PARSE_JSON(RAW_JSON):startTime::TIMESTAMP as event_time,
  PARSE_JSON(RAW_JSON):vehicleCustomer::STRING as customer,
  sig.value:name::STRING as signal_name,
  sig.value:displayName::STRING as signal_display,
  sig.value:values[0]:value::STRING as first_reading
FROM ODOS_EVENTS,
  LATERAL FLATTEN(input => PARSE_JSON(RAW_JSON):signals) sig
LIMIT 20;

## The VIN Join: Connecting CRM to Telemetry

In [ ]:
-- How many VINs overlap between Salesforce and Odos?
SELECT COUNT(DISTINCT a."Chassis_Number__c") as matching_vins
FROM ASSET a
INNER JOIN (
    SELECT DISTINCT PARSE_JSON("RAW_JSON"):vin::STRING as vin
    FROM ODOS_EVENTS
) o ON a."Chassis_Number__c" = o.vin;

In [ ]:
-- Joined view: vehicle details + latest telemetry location
SELECT
    a."Chassis_Number__c" as vin,
    a."Name" as model,
    a."Status" as status,
    a."LatestLocation__Latitude__s" as lat,
    a."LatestLocation__Longitude__s" as lon,
    a."LatestMileage__c" as mileage
FROM ASSET a
WHERE a."Chassis_Number__c" IS NOT NULL
    AND a."LatestLocation__Latitude__s" IS NOT NULL
LIMIT 20;

## Explore Complete!

Key findings:
- 2,492 VINs match between Salesforce and Odos
- Telemetry includes SOC, speed, temperature, GPS coordinates
- Sales pipeline has 1,056 opportunities worth £2.5B+

**Next:** Open `03_build_gold.ipynb` to build the analytics layer.